In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import torch
from omegaconf import OmegaConf

from core.data.mining import MiningConfig, build_cache
from torch.utils.data import DataLoader
from core.data.module import AlignData
from core.data.vocab import TagTokenizer
from core.model.bobert import BobertForAlignment
from core.training.align import setup_alignment, train

config = OmegaConf.load(PROJECT_ROOT / 'config.yaml')
config.alignment.db_path = str(PROJECT_ROOT / 'data' / 'beatmap_dataset175k')
config.alignment.mining_cache_path = str(PROJECT_ROOT / 'data' / 'mining_cache.parquet')
config.alignment.checkpoint_dir = str(PROJECT_ROOT / 'experiments' / 'alignment')
config.model.dim_feedforward = 2048
config.components.compile_model = False
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

In [ ]:
cache_path = Path(config.alignment.mining_cache_path)
BUILD_CACHE = not cache_path.exists()

if BUILD_CACHE:
    cache = build_cache(
        data_dir=PROJECT_ROOT / 'data',
        dataset_dir=PROJECT_ROOT / 'data' / 'beatmap_dataset175k',
        output_path=cache_path,
        config=MiningConfig(
            top_k=32,
            candidate_k=256,
            block_size=128,
            star_radius=0.5,
            max_star_delta=1.0,
            max_ratio_distance=0.35,
            max_anchors=32768,
        ),
    )
else:
    import pandas as pd
    cache = pd.read_parquet(cache_path)

cache.head(), len(cache)

In [ ]:
config.alignment.batch_size = 16
config.alignment.gradient_accumulation_steps = 4
config.alignment.num_epochs = 3

datamodule = AlignData(config, TagTokenizer())
datamodule.prepare_data()
datamodule.setup('fit')

model = BobertForAlignment.from_config(config, device)
model.get_summary()

In [ ]:
PRETRAIN_CKPT = PROJECT_ROOT / "experiments" / "checkpoints" / "last.ckpt"
if not PRETRAIN_CKPT.exists():
    checkpoint_dir = PROJECT_ROOT / "experiments" / "checkpoints"
    candidates = sorted(checkpoint_dir.glob("last*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True)
    candidates = candidates or sorted(checkpoint_dir.glob("*.ckpt"), key=lambda p: p.stat().st_mtime, reverse=True)
    PRETRAIN_CKPT = candidates[0] if candidates else PRETRAIN_CKPT

if PRETRAIN_CKPT:
    if not PRETRAIN_CKPT.exists():
        raise FileNotFoundError(PRETRAIN_CKPT)

    ckpt = torch.load(PRETRAIN_CKPT, map_location='cpu', weights_only=False)
    raw_state = ckpt.get('state_dict', ckpt)
    state = {}
    for key, value in raw_state.items():
        for prefix in ('model._orig_mod.', 'model.', '_orig_mod.'):
            if key.startswith(prefix):
                key = key[len(prefix):]
                break
        if key.startswith('difficulty_head.head.'):
            key = key.replace('difficulty_head.head.', 'difficulty_head.', 1)
        state[key] = value

    model_state = model.state_dict()
    compatible_state = {
        key: value
        for key, value in state.items()
        if key in model_state and tuple(model_state[key].shape) == tuple(value.shape)
    }
    skipped = sorted(set(state) - set(compatible_state))
    missing, unexpected = model.load_state_dict(compatible_state, strict=False)
    print(f'loaded={len(compatible_state)} skipped={len(skipped)} missing={len(missing)} unexpected={len(unexpected)}')
    print(f'checkpoint={PRETRAIN_CKPT}')
    print('loaded difficulty head:', 'difficulty_head.weight' in compatible_state and 'difficulty_head.bias' in compatible_state)


In [ ]:
module, trainer = setup_alignment(config, model, datamodule.normalizer)
train(module, trainer, datamodule)